# Projeto da disciplina CCM-109 - Tópicos especiais de IA - Deep Learning



# Detecção de deepfake

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jtlimo/ccm-109/blob/main/main.ipynb)

---

## Índice

1. [Configuração](#1-configuração)
2. [Funções auxiliares](#2-funções-auxiliares)
3. [Execução pipeline](#3-execução-da-pipeline)

## 1. Configuração

In [ ]:
# @title Instalar dependências

%pip install deepface opencv-python tqdm tf-keras

In [ ]:
# @title Imports

import os
import cv2
import numpy as np
from pathlib import Path
from collections import defaultdict
from tqdm.notebook import tqdm
from deepface import DeepFace

In [ ]:
# @title Configurações do pipeline

VIDEO_PATH = "/dataset/FaceForensics/manipulated_sequences/DeepFakeDetection/c23/videos/01_02__exit_phone_room__YVGY8LOK.mp4"
OUTPUT_DIR = "/dataset/FaceForensics/manipulated_sequences/DeepFakeDetection/c23/frames"

DETECTOR_BACKEND = "retinaface"
CONFIDENCE_THRESHOLD = 0.9
ALIGN = True

SKIP_FRAMES = 0                    # 0 = todos; 4 = pula 4 (processa 1 a cada 5)
MAX_FRAMES_PER_VIDEO = 30          # None = todos; ex: 100 = máx 100 frames salvos
MAX_READ_LIMIT = 300

VIDEO_EXTENSIONS = ("*.mp4", "*.avi", "*.mov", "*.mkv")

## 2. Funções auxiliares

In [ ]:
# @title Extrair rostos de um frame

def extract_faces_from_frame(frame, detector_backend="retinaface",
                             confidence_threshold=0.85, align=True):

    try:
        faces = DeepFace.extract_faces(
            img_path=frame,
            detector_backend=detector_backend,
            enforce_detection=False,
            align=align
        )
    except Exception as e:
        return []

    results = []
    for idx, face_info in enumerate(faces):
        confidence = face_info.get("confidence", 0)
        if confidence >= confidence_threshold:
            face_img = face_info["face"]

            if face_img.dtype in (np.float32, np.float64):
                face_img = (face_img * 255).astype(np.uint8)

            results.append({
                "face": face_img,
                "confidence": confidence,
                "facial_area": face_info.get("facial_area", {}),
                "face_id": idx,
            })

    results.sort(key=lambda x: x["confidence"], reverse=True)
    return results

In [ ]:
# @title Processar vídeo

def process_video(video_path, output_dir,
                             detector="retinaface",
                             confidence=0.85,
                             align=True,
                             skip_frames=0,
                             max_frames=30,
                             max_read_limit=None):

    video_path = Path(video_path)
    video_name = video_path.stem

    frames_dir = Path(output_dir) / video_name
    frames_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return 0


    frame_idx = 0
    saved_frames_count = 0

    while True:
        if max_frames is not None and saved_frames_count >= max_frames:
            break

        if max_read_limit is not None and frame_idx >= max_read_limit:
            break

        ret, frame = cap.read()
        if not ret:
            break

        if skip_frames > 0 and frame_idx % (skip_frames + 1) != 0:
            frame_idx += 1
            continue

        faces = extract_faces_from_frame(
            frame,
            detector_backend=detector,
            confidence_threshold=confidence,
            align=align
        )

        if faces:
            for face_idx, face_data in enumerate(faces):
                filename = f"frame_{frame_idx + 1}_face_{face_idx}.jpg"
                output_path = frames_dir / filename

                face_bgr = cv2.cvtColor(face_data["face"], cv2.COLOR_RGB2BGR)
                cv2.imwrite(str(output_path), face_bgr)

            saved_frames_count += 1

        frame_idx += 1

    cap.release()
    return saved_frames_count

In [ ]:
# @title Processar o dataset

def process_dataset(dataset_root, output_root):
    dataset_root = Path(dataset_root)
    output_root = Path(output_root)

    video_files = []
    for ext in VIDEO_EXTENSIONS:
        video_files.extend(list(dataset_root.rglob(ext)))

    total_videos = len(video_files)
    if total_videos == 0:
        return

    pbar_videos = tqdm(video_files, desc="Progresso Geral", unit="video")

    total_frames_dataset = 0
    processed_videos = 0

    for video_path in pbar_videos:
        relative_path = video_path.parent.relative_to(dataset_root)
        video_output_dir = output_root / relative_path

        pbar_videos.set_postfix({"Vídeo": video_path.name[:25]})

        saved_frames = process_video(
            video_path=video_path,
            output_dir=video_output_dir,
            detector=DETECTOR_BACKEND,
            confidence=CONFIDENCE_THRESHOLD,
            align=ALIGN,
            skip_frames=SKIP_FRAMES,
            max_frames=MAX_FRAMES_PER_VIDEO,
            max_read_limit=MAX_READ_LIMIT
        )

        total_frames_dataset += saved_frames
        processed_videos += 1

## 3. Execução da pipeline

In [ ]:
# @title Executar pipeline de extração de rostos

stats = process_dataset(
    dataset_root=VIDEO_PATH,
    output_root=OUTPUT_DIR
)